# Notebook 04 — Model Evaluation

## What does this notebook do?

We take the best trained model and measure how well it performs.
We do this on TWO sets:

1. VALIDATION SET first — to confirm the model trained correctly
2. TEST SET — the final honest score we report in the paper

## What metrics do we calculate?

| Metric | Simple meaning |
|--------|---------------|
| Sensitivity (Recall) | Of all real malignant cases, how many did we catch? |
| Specificity | Of all non-malignant cases, how many did we correctly leave alone? |
| Precision | Of all cases we called malignant, how many actually were? |
| F1 Score | Balance between precision and recall |
| AUROC | Overall ability to separate classes (1.0 = perfect, 0.5 = random) |
| Accuracy | Overall percentage of correct predictions |

## Important rule — read before running!

Run the TEST SET evaluation ONLY ONCE.
Do not check the test results, adjust the model, then re-test.
The test set must give one honest final result.

---
##  Import Libraries

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F        # for softmax
from pathlib import Path
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# sklearn gives us all the metrics we need
from sklearn.metrics import (
    classification_report,      # precision, recall, f1 per class
    confusion_matrix,           # table of predictions vs true labels
    ConfusionMatrixDisplay,     # helper to plot the confusion matrix
    roc_auc_score,              # AUROC score
    roc_curve                   # ROC curve data points
)

# Set seeds for reproducibility
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Libraries imported!')
print(f'Using device: {device}')

---
## Load the Datasets

We load the val and test sets using the same transform
as in notebook 03 (no augmentation, just resize + normalise).

In [ ]:
# Same normalisation values used during training
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# No augmentation for evaluation — we want honest results
eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# Load datasets
val_dataset  = datasets.ImageFolder('data/processed/val',  transform=eval_transform)
test_dataset = datasets.ImageFolder('data/processed/test', transform=eval_transform)

# Create loaders
# shuffle=False — we do not shuffle during evaluation
val_loader  = DataLoader(val_dataset,  batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Class names — used for labels in charts and tables
CLASS_NAMES = val_dataset.classes   # ['benign', 'malignant', 'normal']

print(f'Classes : {CLASS_NAMES}')
print(f'Val  set: {len(val_dataset)} images')
print(f'Test set: {len(test_dataset)} images')

---
## Reload the Best Trained Model

We rebuild the same ResNet-18 architecture from notebook 03
and load the best saved weights.

In [ ]:
def build_model():
    """
    Rebuild the same architecture used during training.
    ResNet-18 with a 3-class head.
    """
    model    = models.resnet18(weights=None)   # no pretrained weights needed now
    in_feats = model.fc.in_features            # 512
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_feats, 3)
    )
    return model


# Build the model structure
model = build_model().to(device)

# Load the best weights saved during Stage 2 training
checkpoint_path = 'outputs/models/best_model_stage2.pth'
model.load_state_dict(torch.load(checkpoint_path, map_location=device))

# Set to evaluation mode
# This disables dropout so predictions are deterministic
model.eval()

print(f'Model loaded from: {checkpoint_path}')
print('Model is in evaluation mode — ready for predictions')

---
## Prediction Function

This function runs the model on any DataLoader and collects:
- The predicted class for each image
- The true class for each image
- The predicted probability for each class (needed for AUROC and ROC curves)

In [ ]:
def get_predictions(model, loader, device):
    """
    Run the model on all images in the loader.

    Returns three lists:
    - all_preds  : predicted class index for each image  [0, 1, 2, 0, ...]
    - all_labels : true class index for each image       [0, 0, 2, 1, ...]
    - all_probs  : predicted probability for each class  [[0.8, 0.1, 0.1], ...]
    """

    model.eval()

    all_preds  = []
    all_labels = []
    all_probs  = []

    with torch.no_grad():   # no gradients needed during evaluation
        for images, labels in loader:

            images = images.to(device)

            # Forward pass — get raw scores (logits)
            logits = model(images)

            # Convert logits to probabilities using softmax
            # softmax makes all values sum to 1.0
            # Example: [2.1, 0.3, -1.0] -> [0.75, 0.15, 0.10]
            probs = F.softmax(logits, dim=1).cpu().numpy()

            # Predicted class = the index with the highest probability
            preds = logits.argmax(dim=1).cpu().numpy()

            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    # Convert lists to numpy arrays for easier calculation
    return (
        np.array(all_preds),
        np.array(all_labels),
        np.array(all_probs)
    )


print('Prediction function defined!')

---
##  Full Metrics Function

This function takes the predictions and calculates all required metrics.
It also generates Figure 4 (confusion matrix) and Figure 5 (ROC curves).

We will call this function twice:
  - Once for the validation set
  - Once for the test set

In [ ]:
def evaluate(preds, labels, probs, class_names, split_name='Test'):
    """
    Calculate and print all metrics.
    Generate confusion matrix and ROC curve figures.

    preds       : array of predicted class indices
    labels      : array of true class indices
    probs       : array of predicted probabilities per class
    class_names : ['benign', 'malignant', 'normal']
    split_name  : 'Validation' or 'Test' (used in titles)
    """

    print('=' * 60)
    print(f'  RESULTS ON {split_name.upper()} SET')
    print('=' * 60)


    # ── 1. Classification Report ──────────────────────────────
    # Prints precision, recall (sensitivity), f1 for each class
    print('\nClassification Report:')
    print('(recall = sensitivity for each class)\n')
    print(classification_report(labels, preds, target_names=class_names))


    # ── 2. Per-class Sensitivity and Specificity ──────────────
    # These are calculated manually from the confusion matrix
    #
    # For each class:
    #   Sensitivity = TP / (TP + FN)
    #     how many real positives did we catch?
    #   Specificity = TN / (TN + FP)
    #     how many real negatives did we correctly reject?
    #
    # TP = True Positive  (predicted class X, actually class X)
    # FN = False Negative (predicted NOT X, actually class X)
    # TN = True Negative  (predicted NOT X, actually NOT X)
    # FP = False Positive (predicted class X, actually NOT X)

    cm = confusion_matrix(labels, preds)

    print('Per-class Sensitivity and Specificity:')
    print(f'{"Class":<12}  {"Sensitivity":>13}  {"Specificity":>13}')
    print('-' * 42)

    for i, cls in enumerate(class_names):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP     # other values in row i
        FP = cm[:, i].sum() - TP     # other values in column i
        TN = cm.sum() - TP - FN - FP

        sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0

        print(f'{cls:<12}  {sensitivity:>13.3f}  {specificity:>13.3f}')


    # ── 3. Overall Accuracy ───────────────────────────────────
    accuracy = (preds == labels).mean()
    print(f'\nOverall Accuracy : {accuracy:.4f}  ({accuracy*100:.1f}%)')


    # ── 4. Macro AUROC ────────────────────────────────────────
    # AUROC = Area Under the ROC Curve
    # 1.0 = perfect classifier
    # 0.5 = random guessing
    # OvR = One vs Rest: each class is compared against all others
    auroc = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
    print(f'Macro AUROC      : {auroc:.4f}')
    print()


    # ── 5. FIGURE 4: Confusion Matrix ─────────────────────────
    # Rows = true class, Columns = predicted class
    # Diagonal = correct predictions
    # Off-diagonal = mistakes

    fig, ax = plt.subplots(figsize=(6, 5))

    disp = ConfusionMatrixDisplay(
        confusion_matrix = cm,
        display_labels   = [c.capitalize() for c in class_names]
    )
    disp.plot(ax=ax, colorbar=False, cmap='Blues')

    ax.set_title(f'Figure 4: Confusion Matrix — {split_name} Set',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted Class')
    ax.set_ylabel('True Class')

    plt.tight_layout()
    os.makedirs('outputs/figures', exist_ok=True)
    save_path = f'outputs/figures/fig4_confusion_matrix_{split_name.lower()}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure 4 saved -> {save_path}')
    print()


    # ── 6. FIGURE 5: ROC Curves ───────────────────────────────
    # One curve per class
    # X axis = False Positive Rate (1 - Specificity)
    # Y axis = True Positive Rate  (Sensitivity)
    # The closer the curve hugs the top-left corner, the better

    colors = ['#2196F3', '#F44336', '#4CAF50']   # blue, red, green

    fig, ax = plt.subplots(figsize=(7, 6))

    for i, (cls, color) in enumerate(zip(class_names, colors)):

        # One-vs-Rest: treat this class as positive, all others as negative
        binary_labels = (labels == i).astype(int)
        class_probs   = probs[:, i]

        # Calculate ROC curve points
        fpr, tpr, _ = roc_curve(binary_labels, class_probs)

        # Calculate AUROC for this class
        auc = roc_auc_score(binary_labels, class_probs)

        # Plot the curve
        ax.plot(fpr, tpr,
                label = f'{cls.capitalize()}  (AUROC = {auc:.3f})',
                color = color,
                linewidth = 2)

    # Diagonal line = random classifier (AUROC = 0.5)
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUROC = 0.5)')

    ax.set_xlabel('False Positive Rate  (1 - Specificity)', fontsize=11)
    ax.set_ylabel('True Positive Rate  (Sensitivity)',      fontsize=11)
    ax.set_title(f'Figure 5: ROC Curves per Class — {split_name} Set',
                 fontsize=12, fontweight='bold')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = f'outputs/figures/fig5_roc_curves_{split_name.lower()}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure 5 saved -> {save_path}')

    return {
        'accuracy' : accuracy,
        'auroc'    : auroc,
        'cm'       : cm
    }


print('Evaluation function defined!')

---
## Evaluate on VALIDATION SET

We run the model on the validation set first.
This is to confirm training worked correctly.

These results are NOT the final numbers for your report.
They are just a sanity check.

In [ ]:
print('Running predictions on VALIDATION set...')
val_preds, val_labels, val_probs = get_predictions(model, val_loader, device)

# Evaluate and print all metrics
val_results = evaluate(
    preds       = val_preds,
    labels      = val_labels,
    probs       = val_probs,
    class_names = CLASS_NAMES,
    split_name  = 'Validation'
)

---
##  Evaluate on TEST SET

This is the FINAL evaluation.
These are the numbers that go in your report.

Run this cell ONLY ONCE.
Do not re-run after making any changes to the model.

In [ ]:
print('Running predictions on TEST set...')
print('Remember: run this only ONCE!')
print()

test_preds, test_labels, test_probs = get_predictions(model, test_loader, device)

# Evaluate and print all metrics
test_results = evaluate(
    preds       = test_preds,
    labels      = test_labels,
    probs       = test_probs,
    class_names = CLASS_NAMES,
    split_name  = 'Test'
)

---
## Print Clean Metrics Table for Report

This prints a clean formatted table you can copy
directly into your report as Table 2.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

# Get precision, recall, f1 for each class
precision, recall, f1, support = precision_recall_fscore_support(
    test_labels,
    test_preds,
    labels = [0, 1, 2]
)

# Get per-class AUROC
per_class_auroc = []
for i in range(len(CLASS_NAMES)):
    binary = (test_labels == i).astype(int)
    auc    = roc_auc_score(binary, test_probs[:, i])
    per_class_auroc.append(auc)

# Get per-class sensitivity and specificity from confusion matrix
cm = confusion_matrix(test_labels, test_preds)
sensitivities  = []
specificities  = []

for i in range(len(CLASS_NAMES)):
    TP = cm[i, i]
    FN = cm[i, :].sum() - TP
    FP = cm[:, i].sum() - TP
    TN = cm.sum() - TP - FN - FP
    sensitivities.append(TP / (TP + FN) if (TP + FN) > 0 else 0)
    specificities.append(TN / (TN + FP) if (TN + FP) > 0 else 0)


# Print the table
print('=' * 80)
print('  TABLE 2: Test Set Performance Metrics  (copy this into your report)')
print('=' * 80)
print(f'{"Class":<12}  {"Sensitivity":>12}  {"Specificity":>12}  '
      f'{"Precision":>10}  {"F1":>8}  {"AUROC":>8}  {"N":>5}')
print('-' * 80)

for i, cls in enumerate(CLASS_NAMES):
    print(f'{cls:<12}  '
          f'{sensitivities[i]:>12.3f}  '
          f'{specificities[i]:>12.3f}  '
          f'{precision[i]:>10.3f}  '
          f'{f1[i]:>8.3f}  '
          f'{per_class_auroc[i]:>8.3f}  '
          f'{support[i]:>5}')

print('-' * 80)
overall_acc   = (test_preds == test_labels).mean()
overall_auroc = roc_auc_score(test_labels, test_probs,
                               multi_class='ovr', average='macro')
print(f'{"Overall":<12}  '
      f'{"":>12}  '
      f'{"":>12}  '
      f'{"":>10}  '
      f'{"":>8}  '
      f'{overall_auroc:>8.3f}  '
      f'{len(test_labels):>5}')
print(f'\nOverall Accuracy : {overall_acc:.3f}  ({overall_acc*100:.1f}%)')
print(f'Macro AUROC      : {overall_auroc:.3f}')
print('=' * 80)

---
##  Show Some Example Predictions

We display a few test images with their true label
and what the model predicted.

Green border = correct prediction
Red border   = wrong prediction

In [ ]:
from PIL import Image
import random

random.seed(42)

# Collect some example images from the test set
# We get the file paths directly from the dataset
sample_paths  = [test_dataset.samples[i][0] for i in range(len(test_dataset))]
sample_labels = [test_dataset.samples[i][1] for i in range(len(test_dataset))]

# Pick 8 random examples
indices = random.sample(range(len(test_dataset)), 8)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('Example Predictions on Test Set\n'
             '(Green = Correct   Red = Wrong)',
             fontsize=12, fontweight='bold')

for ax, idx in zip(axes.flat, indices):

    # Open and display the image
    img        = Image.open(sample_paths[idx]).convert('L')
    true_label = sample_labels[idx]
    pred_label = test_preds[idx]

    ax.imshow(img, cmap='gray')
    ax.axis('off')

    # Choose border colour based on correct/wrong prediction
    border_color = 'green' if pred_label == true_label else 'red'
    for spine in ax.spines.values():
        spine.set_edgecolor(border_color)
        spine.set_linewidth(4)
        spine.set_visible(True)

    # Title shows true and predicted class
    ax.set_title(
        f'True : {CLASS_NAMES[true_label]}\n'
        f'Pred : {CLASS_NAMES[pred_label]}',
        fontsize=9,
        color = 'green' if pred_label == true_label else 'red'
    )

plt.tight_layout()
plt.savefig('outputs/figures/example_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Example predictions saved -> outputs/figures/example_predictions.png')

---
##  Final Summary

In [ ]:
print('=' * 55)
print('  EVALUATION COMPLETE')
print('=' * 55)
print(f'  Overall Accuracy : {overall_acc:.3f}')
print(f'  Macro AUROC      : {overall_auroc:.3f}')
print()
print('  Figures saved:')
print('    outputs/figures/fig4_confusion_matrix_test.png  -> Figure 4')
print('    outputs/figures/fig5_roc_curves_test.png        -> Figure 5')
print('    outputs/figures/example_predictions.png')
print()
print('  Use the Table 2 output above in your report Results section.')
print()
print('  NEXT STEP: Run 05_GradCAM.ipynb')
print('  That notebook shows WHICH part of the image the model looked at.')